# Method 2 completion 3: round2
Attach the strict completion bundle. See docs/method2_completion.md for the required previous outputs. GPU T4, fresh session. Test results must not select hyperparameters.


In [1]:
from pathlib import Path
import os, shutil, subprocess, sys, json
from pathlib import Path
import hashlib, json

def file_digest(path: Path) -> str:
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def choose_bundle(input_root: Path) -> Path:
    candidates = []
    for manifest in sorted(input_root.rglob("completion_manifest.json")):
        data = json.loads(manifest.read_text(encoding="utf-8"))
        if data.get("files") and all((manifest.parent / name).is_file() for name in data["files"]):
            candidates.append(manifest)
    if not candidates or len({file_digest(p) for p in candidates}) != 1:
        raise RuntimeError(f"Attach one complete bundle identity. Complete candidates: {[str(p) for p in candidates]}")
    print("Bundle selected:", candidates[0].parent)
    return candidates[0].parent

def choose_stage(input_root: Path, stage: str, bundle_digest: str) -> Path:
    candidates = []
    identities = set()
    for marker in sorted(input_root.rglob(f"{stage}_completed.json")):
        if marker.parts[-4:-1] != ("results", "method2", "completion"):
            continue
        root = marker.parents[3]
        data = json.loads(marker.read_text(encoding="utf-8"))
        hashes = data.get("checkpoint_hashes", {})
        if data.get("stage") != stage or data.get("completed") is not True or not hashes:
            continue
        if data.get("bundle_manifest_sha256") != bundle_digest:
            print("Skipping marker from another bundle:", marker)
            continue
        invalid = [name for name, digest in hashes.items()
                   if not (root / name).is_file() or file_digest(root / name) != digest]
        if invalid:
            print("Skipping incomplete or mismatched stage output:", marker, invalid[:3])
            continue
        candidates.append(marker)
        identities.add(json.dumps(hashes, sort_keys=True))
    if not candidates or len(identities) != 1:
        raise RuntimeError(f"Attach one complete {stage} checkpoint identity. Verified candidates: {[str(p) for p in candidates]}")
    print(f"{stage} input selected:", candidates[0])
    return candidates[0]


os.environ["CUDA_VISIBLE_DEVICES"] = "0"
bundle = choose_bundle(Path("/kaggle/input"))
work = Path("/kaggle/working")
for name in ["src", "scripts", "configs", "data", "docs", "notebooks"]:
    shutil.copytree(bundle / name, work / name, dirs_exist_ok=True)
shutil.copy2(bundle / "same_domain_feasibility.json", work / "same_domain_feasibility.json")
shutil.copy2(bundle / "completion_manifest.json", work / "completion_manifest.json")
os.chdir(work)
caches = list(Path("/kaggle/input").rglob("models--BAAI--bge-m3"))
if caches:
    cache_hub = caches[0].parent
    os.environ["HF_HUB_CACHE"] = str(cache_hub)
    os.environ["HF_HOME"] = str(cache_hub.parent)
pins = json.loads(Path("configs/method2/pinned_versions.json").read_text())["pinned"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[f"{k}=={v}" for k,v in pins.items()], "jsonschema", "pyyaml", "matplotlib", "rank_bm25", "datasets", "accelerate"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
lora_smoke = """import torch
from transformers import XLMRobertaConfig, XLMRobertaModel
from peft import LoraConfig
config = XLMRobertaConfig(vocab_size=32, hidden_size=8, num_hidden_layers=1, num_attention_heads=2, intermediate_size=16, max_position_embeddings=32)
model = XLMRobertaModel(config)
model.add_adapter(LoraConfig(r=2, lora_alpha=4, target_modules=["query", "value"]))
model(torch.tensor([[0, 5, 2]])).last_hidden_state.square().mean().backward()
assert any(p.grad is not None for n, p in model.named_parameters() if "lora_" in n)
print("LoRA environment smoke passed")
"""
subprocess.run([sys.executable, "-c", lora_smoke], check=True)


Bundle selected: /kaggle/input/datasets/dathq12/output-method2-completion-02-round1/output_method2-completion-02-round1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.6/739.6 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 35.7 MB/s eta 0:00:00
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
LoRA environment smoke passed


CompletedProcess(args=['/usr/bin/python3', '-c', 'import torch\nfrom transformers import XLMRobertaConfig, XLMRobertaModel\nfrom peft import LoraConfig\nconfig = XLMRobertaConfig(vocab_size=32, hidden_size=8, num_hidden_layers=1, num_attention_heads=2, intermediate_size=16, max_position_embeddings=32)\nmodel = XLMRobertaModel(config)\nmodel.add_adapter(LoraConfig(r=2, lora_alpha=4, target_modules=["query", "value"]))\nmodel(torch.tensor([[0, 5, 2]])).last_hidden_state.square().mean().backward()\nassert any(p.grad is not None for n, p in model.named_parameters() if "lora_" in n)\nprint("LoRA environment smoke passed")\n'], returncode=0)

In [2]:
prior_stage = 'round1'
if prior_stage:
    prior_marker = choose_stage(Path("/kaggle/input"), prior_stage, file_digest(work / "completion_manifest.json"))
    previous = prior_marker.parents[3]
    for name in ["artifacts/method2", "results/method2/completion", "data/method2/index"]:
        if (previous / name).exists():
            shutil.copytree(previous / name, work / name, dirs_exist_ok=True, ignore=shutil.ignore_patterns("checkpoint-*"))
if False:
    validation_marker = choose_stage(Path("/kaggle/input"), "validation", file_digest(work / "completion_manifest.json"))
    validation_root = validation_marker.parents[3]
    shutil.copytree(validation_root / "results/method2/completion/validation", work / "results/method2/completion/validation", dirs_exist_ok=True)
    shutil.copy2(validation_marker, work / "results/method2/completion/validation_completed.json")
    shutil.copytree(validation_root / "artifacts/method2/crossencoder", work / "artifacts/method2/crossencoder", dirs_exist_ok=True, ignore=shutil.ignore_patterns("checkpoint-*"))
ce = Path("artifacts/method2/crossencoder/run01/final")
if False and not (ce / "crossencoder_heads.pt").exists():
    heads = [p for p in Path("/kaggle/input").rglob("crossencoder_heads.pt") if p.parent.name == "final"]
    if len(heads) != 1:
        raise RuntimeError(f"Attach one CE final checkpoint (model + heads + tokenizer), found {len(heads)}")
    shutil.copytree(heads[0].parent, ce, dirs_exist_ok=True)


Skipping incomplete or mismatched stage output: /kaggle/input/datasets/dathq12/output-method2-completion-02-round1/output_method2-completion-02-round1/method2_completion_round1_reports/results/method2/completion/round1_completed.json ['artifacts/method2/biencoder/strict_round1/final/modules.json', 'artifacts/method2/biencoder/strict_round1/final/adapter_model.safetensors', 'artifacts/method2/biencoder/strict_round1/final/sentence_bert_config.json']
round1 input selected: /kaggle/input/datasets/dathq12/output-method2-completion-02-round1/output_method2-completion-02-round1/results/method2/completion/round1_completed.json


In [3]:
subprocess.run([sys.executable, "scripts/method2/run_completion.py", "round2"], check=True)
print(Path("results/method2/completion/round2_completed.json").read_text(encoding="utf-8")[:2000])


[biencoder] TensorFlow version 2.20.0 available.
[biencoder] JAX version 0.7.2 available.
[biencoder] No device provided, using cuda:0
[biencoder] Loading SentenceTransformer model from artifacts/method2/biencoder/strict_round1/final.
[biencoder] HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[biencoder] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
[biencoder] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/config.json "HTTP/1.1 200 OK"
[biencoder] HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/config.json "HTTP/1.1 200 OK"
[biencoder] HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[biencoder] HTTP Request: HEAD https://hug

{
  "n_rows": 94634,
  "n_mined": 78435,
  "output": "data/method2/biencoder/train_mined.jsonl"
}


[biencoder] CUDA_VISIBLE_DEVICES=0 — ép chạy một GPU
[biencoder] TensorFlow version 2.20.0 available.
[biencoder] JAX version 0.7.2 available.
/kaggle/working/src/models/biencoder/train.py:223: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses import CachedMultipleNegativesRankingLoss
[biencoder] train pairs: 78435
[biencoder] No device provided, using cuda:0
[biencoder] HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
[biencoder] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
[biencoder] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/modules.json "HTTP/1.1 200 OK"
[biencoder] HTTP Req

{'loss': '3.897', 'grad_norm': '2.88', 'learning_rate': '1.054e-05', 'epoch': '0.1629'}
{'loss': '3.032', 'grad_norm': '1.772', 'learning_rate': '2e-05', 'epoch': '0.3257'}


 22%|██▏       | 200/921 [2:26:28<8:53:10, 44.37s/it][biencoder] Saving model checkpoint to artifacts/method2/biencoder/strict_round2/checkpoint-200
[biencoder] Saving model to artifacts/method2/biencoder/strict_round2/checkpoint-200
[biencoder] HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[biencoder] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/config.json "HTTP/1.1 200 OK"
[biencoder] HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[biencoder] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/config.json "HTTP/1.1 200 OK"

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 20.96it/s]


...

{
  "tau": 0.35,
  "tau_call": 0.35,
  "k_max": 3,
  "gap_delta": 0.21,
  "strategy": "gap",
  "calibrated_on": "data/method2/biencoder/val.jsonl",
  "metrics": {
    "abstention": {
      "macro_f1": 0.9082,
      "f1_call": 0.9804,
      "f1_no_call": 0.836,
      "negative_recall": 0.8636,
      "call_recall": 0.9767
    },
    "call_selection": {
      "absolute": {
        "value": 0.35,
        "f1": 0.9482,
        "precision": 0.9467,
        "recall": 0.9497,
        "tool_set_accuracy": 0.8837
      },
      "gap": {
        "value": 0.21,
        "f1": 0.9552,
        "precision": 0.9523,
        "recall": 0.9583,
        "tool_set_accuracy": 0.8972
      },
      "winner": "gap"
    },
    "retrieval": {
      "n_positive_samples": 7521,
      "n_gold_tools": 9846,
      "mrr": 0.9966,
      "ndcg@10": 0.9968,
      "micro_recall@1": 0.7596,
      "full_recall@1": 0.7246,
      "micro_recall@3": 0.9938,
      "full_recall@3": 0.992,
      "micro_recall@5": 0.9999,
      "fu

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2493.41it/s]


{
  "n_positive_samples": 7521,
  "n_gold_tools": 9846,
  "mrr": 0.9966,
  "ndcg@10": 0.9968,
  "micro_recall@1": 0.7596,
  "full_recall@1": 0.7246,
  "micro_recall@3": 0.9938,
  "full_recall@3": 0.992,
  "micro_recall@5": 0.9999,
  "full_recall@5": 0.9999,
  "micro_recall@10": 1.0,
  "full_recall@10": 1.0
}


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2406.68it/s]


{
  "n_positive_samples": 7521,
  "n_gold_tools": 9846,
  "mrr": 0.8312,
  "ndcg@10": 0.8341,
  "micro_recall@1": 0.5717,
  "full_recall@1": 0.5321,
  "micro_recall@3": 0.7988,
  "full_recall@3": 0.7748,
  "micro_recall@5": 0.8599,
  "full_recall@5": 0.8396,
  "micro_recall@10": 0.9173,
  "full_recall@10": 0.9019
}
[run_manifest] → artifacts/method2/biencoder/strict_round2/run_manifest.json
[run_manifest] commit=6eb1b582b4eaba9ff8a9b51c4d88ae5bd0898465 dirty=False
[run_manifest] overlap sau decontamination: {'test∩train': 0, 'test∩val': 0, 'train∩val': 0}
[run_manifest] OK: đủ toàn bộ mục audit
{
  "stage": "round2",
  "completed": true,
  "bundle_manifest_sha256": "d562aa4b38bbba20eb28adfc1702df42c193add4795c7415528a62bde945c908",
  "device": "Tesla T4",
  "torch": "2.10.0+cu128",
  "warmup": "3 per evaluation mode; excluded from measurements",
  "checkpoint_hashes": {
    "artifacts/method2/biencoder/strict_round1/final/README.md": "8d54d696f1277e3cce3405d478f0bf361e1ec397c79650655da

In [4]:
archive = work / "method2_completion_round2_reports.tar.gz"
items = [name for name in ["results/method2/completion", "data/method2/biencoder/train_mined.jsonl"] if Path(name).exists()]
subprocess.run(["tar", "czf", str(archive), *items], check=True)
print(archive, archive.stat().st_size)
print("Save the whole notebook output for the next stage; this small archive does not contain model weights or the index.")


/kaggle/working/method2_completion_round2_reports.tar.gz 7382964
Save the whole notebook output for the next stage; this small archive does not contain model weights or the index.
